In [7]:
# Código en R
j_eval <- function(cmd) {
  .ensure_julia_ready()
  lineas <- strsplit(cmd, "\n")[[1]]
  buffer <- ""; en_bloque <- 0; en_comillas <- 0; resultado_final <- NULL

  for (l in lineas) {
    if (trimws(l) == "") next
    buffer <- paste0(buffer, l, "\n")
    
    # --- Tu lógica estable de detección ---
    en_comillas <- (en_comillas + lengths(regmatches(l, gregexpr('"', l, fixed = TRUE)))) %% 2 
    if (en_comillas == 0) {
      abrir <- grepl("\\b(do|function|for|if|begin|let|while)\\b", l) + 
               lengths(regmatches(l, gregexpr("(", l, fixed = TRUE))) + 
               lengths(regmatches(l, gregexpr("[", l, fixed = TRUE)))
      cerrar <- grepl("\\bend\\b", l) + 
                lengths(regmatches(l, gregexpr(")", l, fixed = TRUE))) + 
                lengths(regmatches(l, gregexpr("]", l, fixed = TRUE)))
      en_bloque <- en_bloque + abrir - cerrar
    }
    
    if (en_bloque <= 0 && en_comillas == 0) {
      res_raw <- JuliaConnectoR::juliaCall("Main._unal_core_executor", buffer, FALSE, "", 72, 800, 500, 12)
      
      # --- AJUSTE DE SALIDA: Forzar nueva línea ---
      res_l <- gsub("\033\\[[0-9;]*m", "", res_raw) # Limpiar ANSI
      
      # Separamos por líneas para procesar el prompt
      partes <- strsplit(res_l, "\n")[[1]]
      for (p in partes) {
        if (grepl("^julia> ", p)) {
          # Si el comando y el resultado están pegados (separados por 2+ espacios), los dividimos
          p_div <- gsub("^(julia> .*?)\\s{2,}(?=\\S)", "\\1\n", p, perl = TRUE)
          cat(p_div, "\n")
        } else {
          cat(p, "\n")
        }
      }
      # ---------------------------------------------
      
      lineas_res <- strsplit(res_raw, "\n")[[1]]
      lineas_res <- trimws(lineas_res[lineas_res != ""])
      temp_res <- tail(lineas_res[!grepl("^julia>", lineas_res)], 1)
      if (length(temp_res) > 0) resultado_final <- temp_res
      buffer <- ""; en_bloque <- 0; en_comillas <- 0
    }
  }
  if (is.null(resultado_final)) return(invisible(NULL))
  val_limpio <- gsub('"', '', resultado_final)
  num_val <- suppressWarnings(as.numeric(val_limpio))
  return(if (!is.na(num_val)) num_val else val_limpio)
}

j_plot <- function(cmd, n = "tmp_plot.png", dpi = 300, w = 800, h = NULL, ratio = 1.6, fontsize = 12) {
  .ensure_julia_ready()
  if (is.null(h)) h <- round(w / ratio)
  lineas <- strsplit(cmd, "\n")[[1]]
  buffer <- ""; en_bloque <- 0; en_comillas <- 0
  
  for (i in 1:length(lineas)) {
    l <- lineas[i]
    if (trimws(l) == "") next
    buffer <- paste0(buffer, l, "\n")
    en_comillas <- (en_comillas + lengths(regmatches(l, gregexpr('"', l, fixed = TRUE)))) %% 2 
    
    if (en_comillas == 0) {
      abrir <- grepl("\\b(do|function|for|if|begin|let|while)\\b", l) + 
               lengths(regmatches(l, gregexpr("(", l, fixed = TRUE))) + 
               lengths(regmatches(l, gregexpr("[", l, fixed = TRUE)))
      cerrar <- grepl("\\bend\\b", l) + 
                lengths(regmatches(l, gregexpr(")", l, fixed = TRUE))) + 
                lengths(regmatches(l, gregexpr("]", l, fixed = TRUE)))
      en_bloque <- en_bloque + abrir - cerrar
    }
    
    if (en_bloque <= 0 && en_comillas == 0) {
      es_ultimo <- (i == length(lineas))
      log_out <- JuliaConnectoR::juliaCall("_unal_core_executor", buffer, es_ultimo, n, dpi, as.integer(w), as.integer(h), as.integer(fontsize))
      
      # Mismo ajuste de salida para j_plot
      log_l <- gsub("\033\\[[0-9;]*m", "", log_out)
      partes_log <- strsplit(log_l, "\n")[[1]]
      for (p in partes_log) {
        if (grepl("^julia> ", p)) {
          cat(gsub("^(julia> .*?)\\s{2,}(?=\\S)", "\\1\n", p, perl = TRUE), "\n")
        } else {
          cat(p, "\n")
        }
      }
      buffer <- ""; en_bloque <- 0; en_comillas <- 0
    }
  }
  if (file.exists(n)) {
    img <- png::readPNG(n)
    grid::grid.newpage()
    grid::grid.raster(img)
  }
}

In [12]:
# Código en R

library(starsdata)
library(terra)
library(stars)
library(reticulate)

# 1. Localización del ZIP dentro del paquete starsdata
f <- "sentinel/S2A_MSIL1C_20180220T105051_N0206_R051_T32ULE_20180221T134037.zip"
granule <- system.file(file = f, package = "starsdata")
granule

# 2. Construcción de la ruta Virtual de GDAL (/vsizip/)
# Rompemos la cadena en varias líneas para que LaTeX pueda procesarla
base_name <- strsplit(basename(granule), ".zip")[[1]]
base_name


# Esta ruta permite leer directamente el XML de metadatos dentro del ZIP sin descomprimir.
#s2_path <- paste0("SENTINEL2_L1C:/vsizip/", granule, "/", base_name, ".SAFE/MTD_MSIL1C.xml:10m:EPSG_32632")
s2_path <- paste0(
  "SENTINEL2_L1C:/vsizip/", 
  granule, 
  "/", 
  base_name, 
  ".SAFE/MTD_MSIL1C.xml:10m:EPSG_32632"
)
#s2_path

# Mostramos el resultado sin comillas ni índices [1] para el libro
cat("Ruta generada:", s2_path, fill = TRUE)

# Guardamos la ruta en un archivo compartido para que Python y Julia la lean
writeLines(s2_path, "s2_shared_path.txt")

terra 1.8.93


Attaching package: ‘terra’


The following object is masked from ‘package:grid’:

    depth


Loading required package: abind

Loading required package: sf

Linking to GEOS 3.12.1, GDAL 3.8.4, PROJ 9.4.0; sf_use_s2() is TRUE



[1] "/usr/local/lib/R/site-library/starsdata/sentinel/S2A_MSIL1C_20180220T105051_N0206_R051_T32ULE_20180221T134037.zip"

[1] "S2A_MSIL1C_20180220T105051_N0206_R051_T32ULE_20180221T134037"

Ruta generada: 
SENTINEL2_L1C:/vsizip//usr/local/lib/R/site-library/starsdata/sentinel/S2A_MSIL1C_20180220T105051_N0206_R051_T32ULE_20180221T134037.zip/S2A_MSIL1C_20180220T105051_N0206_R051_T32ULE_20180221T134037.SAFE/MTD_MSIL1C.xml:10m:EPSG_32632
